<a href="https://colab.research.google.com/github/Shruti-DevAI/Shruti-DevAI/blob/main/Copy_of_chatbot_using_nlp_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("hi")

hi


In [ ]:
# Step 2: Select Basic Libraries
import nltk
import random
import logging

# Download NLTK data (needed for tokenization)
try:
    nltk.download('punkt')  # For splitting text into words
    nltk.download('wordnet')  # For finding base forms of words (e.g., "running" -> "run")
    nltk.download('punkt_tab') # Required for word_tokenize
except Exception as e:
    print(f"Error downloading NLTK data: {e}")
    raise

# Set up logging for company monitoring
logging.basicConfig(filename='company_chatbot.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logging.info("Chatbot initialized")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:

# Step 3: Design Conversation Flow
# Define intents and their keywords (patterns) for matching
intents = [
    {"intent": "greeting", "patterns": ["hi", "hello", "hey", "good morning", "good afternoon"]},
    {"intent": "inquiry", "patterns": ["what is", "tell me about", "how does", "can you explain"]},
    {"intent": "support", "patterns": ["help", "issue", "problem", "support"]},
    {"intent": "product_info", "patterns": ["price of smartwatch" , "earbuds" , "headphones"]},
    {"intent": "order_status", "patterns": ["track my order", "order status", "where is my order"]},
    {"intent": "farewell", "patterns": ["bye", "goodbye", "see you", "thanks"]}
]

# Define responses for each intent
responses = {
    "greeting": [
        "Hello! I’m Mcqueen, your TechTrend Innovations assistant. How can I help you today?",
        "Hi! Mcqueen here for TechTrend Innovations. What’s up?",
        "Hey there! Mcqueen’s ready to assist with your tech needs!"
    ],
   "inquiry": [
        "Mcqueen’s on it! Can you provide more details?",
        "Let Mcqueen look that up for you. What specifically do you want to know?",
        "Interesting question! Could you clarify for Mcqueen?"
    ],
    "support": [
        "Mcqueen’s here to help with TechTrend products. What’s the issue?",
        "Please describe your problem, and Mcqueen will assist!",
        "Let’s sort this out! What’s troubling you?"
    ],
    "farewell": [
        "Thanks for chatting with Mcqueen! Have a great day!",
        "Goodbye from TechTrend Innovations! Come back soon!",
        "Mcqueen says: See you later!"
    ],
    "product_info": [
        "Mcqueen here! Our smartwatch starts at $99. Want more details?",
        "TechTrend’s wireless earbuds are $49. Interested in features?",
        "Our gadgets are top-notch! Which product do you want to know about?"
    ],
    "order_status": [
        "Mcqueen can help track your order! Please provide your order number.",
        "Let Mcqueen check your order status. Got an order ID?",
        "Your order’s on Mcqueen’s radar! Share your order details."
    ],
    "unknown": [
        "Sorry, Mcqueen didn’t catch that. Could you rephrase?",
        "Mcqueen’s confused! Can you clarify your request?",
        "Not sure what you mean. Try again for Mcqueen, please!"
    ]
}

In [ ]:

# Step 4: Prepare Training Data
def prepare_training_data():
    """
    Process the intents to make patterns ready for matching.
    Returns a list of (intent, pattern) tuples with patterns in lowercase.
    """
    training_pairs = []
    for intent_data in intents:
        intent = intent_data["intent"]
        for pattern in intent_data["patterns"]:
            # Convert patterns to lowercase for consistent matching
            training_pairs.append((intent, pattern.lower()))
    return training_pairs

# Test the training data preparation
training_data = prepare_training_data()
logging.info("Training data prepared: %s", training_data)
print("Training data prepared. Example:", training_data[:3])


Training data prepared. Example: [('greeting', 'hi'), ('greeting', 'hello'), ('greeting', 'hey')]


In [ ]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [ ]:
def preprocess_text(text):
    """
    Tokenizes and lemmatizes the input text.
    """
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text.lower())
    lemmas = [lemmatizer.lemmatize(token) for token in tokens]
    return lemmas

In [ ]:
def classify_intent(user_input):
    """
    Classifies the user's input into a predefined intent based on keyword matching.
    """
    user_lemmas = preprocess_text(user_input)
    logging.info(f"User input lemmas: {user_lemmas}")

    # Iterate through each intent and its patterns
    for intent_data in intents:
        intent = intent_data["intent"]
        for pattern in intent_data["patterns"]:
            pattern_lemmas = preprocess_text(pattern)

            # Simple keyword matching: check if all pattern lemmas are in user lemmas
            # This can be improved with more sophisticated matching algorithms (e.g., TF-IDF, word embeddings)
            if all(p_lemma in user_lemmas for p_lemma in pattern_lemmas):
                logging.info(f"Classified intent '{intent}' for input: '{user_input}'")
                return intent

    # If no intent is matched
    logging.info(f"No intent classified for input: '{user_input}'. Returning 'unknown'.")
    return "unknown"

In [ ]:
# Step 6: Add Context Handling
class ContextManager:
    def __init__(self):
        self.previous_intent = None

    def update_context(self, intent):
        """
        Update the previous intent for context tracking.
        """
        self.previous_intent = intent
        logging.info(f"Context updated: Previous intent = {intent}")

    def get_response(self, intent):
        """
        Get a response based on the current intent and previous intent (if applicable).
        """
        # Context-aware response for inquiry after greeting
        if intent == "inquiry" and self.previous_intent == "greeting":
            return random.choice([
                "Great question after saying hi! Can you share more details?",
                "Thanks for asking! What specifically would you like to know?",
                "Great question after saying hi! Mcqueen’s ready to dive in. What’s the details?",
                "Thanks for asking! Mcqueen’s curious—what do you want to know?"
            ])
        # Default response for the intent
        return random.choice(responses.get(intent, responses["unknown"]))


# Test context handling

context_manager = ContextManager()
test_inputs = ["hello", "what is your product?"]
for user_input in test_inputs:
    intent =  classify_intent(user_input)
    response = context_manager.get_response(intent)
    context_manager.update_context(intent)
    print(f"Input: {user_input}\nIntent: {intent}\nResponse: {response}\n")

Input: hello
Intent: greeting
Response: Hi! Mcqueen here for TechTrend Innovations. What’s up?

Input: what is your product?
Intent: inquiry
Response: Thanks for asking! Mcqueen’s curious—what do you want to know?



In [ ]:
# Step 7: Implement Error Handling
def chatbot_response(user_input, context_manager):
    """
    Process user input, classify intent, and return a response.
    Handles errors and logs interactions.
    """
    try:
        # Check for empty or invalid input
        if not user_input or not isinstance(user_input, str):
            logging.warning("Invalid input received: %s", user_input)
            return random.choice(responses["unknown"])

        # Classify intent and get response
        intent = classify_intent(user_input)
        response = context_manager.get_response(intent)
        context_manager.update_context(intent)

        # Log the interaction
        logging.info(f"User input: {user_input} | Intent: {intent} | Response: {response}")
        return response

    except Exception as e:
        logging.error(f"Error in chatbot_response: {e}")
        return random.choice(responses["unknown"])

# Test error handling
context_manager = ContextManager()
test_inputs = ["hello", "what is your product?", "help me", "", "random123"]
for user_input in test_inputs:
    response = chatbot_response(user_input, context_manager)
    print(f"Input: {user_input}\nResponse: {response}\n")

Input: hello
Response: Hi! Mcqueen here for TechTrend Innovations. What’s up?

Input: what is your product?
Response: Thanks for asking! What specifically would you like to know?

Input: help me
Response: Mcqueen’s here to help with TechTrend products. What’s the issue?

Input: 
Response: Sorry, Mcqueen didn’t catch that. Could you rephrase?

Input: random123
Response: Mcqueen’s confused! Can you clarify your request?



In [ ]:
# Step 8: Test and Validate
def test_chatbot():
    """
    Test the chatbot with various inputs to validate functionality.
    """
    print("Running chatbot tests...")
    context_manager = ContextManager()
    test_inputs = [
        "hello",                    # Valid greeting
        "what is your product?",    # Valid inquiry (tests context after greeting)
        "help me",                  # Valid support request
        "bye",                      # Valid farewell
        "price of smartwatch",      # Valid product_info
        "track my order",           # Valid order_status
        "",                         # Empty input (error case)
        "random123 gibberish"       # Invalid input
    ]

    for user_input in test_inputs:
        response = chatbot_response(user_input, context_manager)
        print(f"Input: '{user_input}'")
        print(f"Response: {response}\n")

# Run the tests
test_chatbot()

Running chatbot tests...
Input: 'hello'
Response: Hello! I’m Mcqueen, your TechTrend Innovations assistant. How can I help you today?

Input: 'what is your product?'
Response: Thanks for asking! Mcqueen’s curious—what do you want to know?

Input: 'help me'
Response: Mcqueen’s here to help with TechTrend products. What’s the issue?

Input: 'bye'
Response: Mcqueen says: See you later!

Input: 'price of smartwatch'
Response: Our gadgets are top-notch! Which product do you want to know about?

Input: 'track my order'
Response: Let Mcqueen check your order status. Got an order ID?

Input: ''
Response: Sorry, Mcqueen didn’t catch that. Could you rephrase?

Input: 'random123 gibberish'
Response: Sorry, Mcqueen didn’t catch that. Could you rephrase?



In [ ]:
pip install flask


In [ ]:
# Step 9: Deploy in Company Environment
def main():
    """
    Run an interactive command-line interface for Mcqueen.
    """
    print("Welcome to Mcqueen, your TechTrend Innovations Chatbot! Type 'exit' to quit.")
    context_manager = ContextManager()
    while True:
        user_input = input("You can ask me anything  : ")
        if user_input.lower() == "exit":
            print("Mcqueen's signing off. Goodbye see you soon !")
            logging.info("Chatbot session ended")
            break
        response = chatbot_response(user_input, context_manager)
        print(f"Mcqueen: {response}\n")

# Run tests and start the chatbot
if __name__ == "__main__":
    test_chatbot()  # Run validation tests first
    main()          # Start interactive interface

# Optional: For web deployment, use Flask (uncomment to use)

from flask import Flask, request, jsonify
app = Flask(__name__)
context_manager = ContextManager()

@app.route('/chat', methods=['POST'])
def chat():
    user_input = request.json.get('message', '')
    response = chatbot_response(user_input, context_manager)
    return jsonify({'response': response})

if __name__ == "__main__":
    app.run(host='0.0.0.0', port=5000)

if __name__ == "__main__":
    test_chatbot()  # Run tests to validate before deployment
    main()  # Start interactive interface

Running chatbot tests...
Input: 'hello'
Response: Hey there! Mcqueen’s ready to assist with your tech needs!

Input: 'what is your product?'
Response: Thanks for asking! Mcqueen’s curious—what do you want to know?

Input: 'help me'
Response: Let’s sort this out! What’s troubling you?

Input: 'bye'
Response: Goodbye from TechTrend Innovations! Come back soon!

Input: 'price of smartwatch'
Response: Our gadgets are top-notch! Which product do you want to know about?

Input: 'track my order'
Response: Your order’s on Mcqueen’s radar! Share your order details.

Input: ''
Response: Mcqueen’s confused! Can you clarify your request?

Input: 'random123 gibberish'
Response: Not sure what you mean. Try again for Mcqueen, please!

Welcome to Mcqueen, your TechTrend Innovations Chatbot! Type 'exit' to quit.
You can ask me anything  : hello
Mcqueen: Hi! Mcqueen here for TechTrend Innovations. What’s up?

You can ask me anything  : tell me about your product 
Mcqueen: Thanks for asking! What specifi